# 04 Horizon Dataset Audit

This notebook validates the training datasets generated for live and fixed
race-progress horizons. Fixed horizons should keep one row per driver-session,
while `live_any_lap` keeps all eligible laps after minimum history rules.

In [1]:
from pathlib import Path
import json
from datetime import datetime

import numpy as np
import pandas as pd
import plotly.express as px

ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

NOTEBOOK_NAME = "04_horizon_dataset_audit"
GOLD = ROOT / "data" / "gold"
FEATURES = GOLD / "features"
LABELS = GOLD / "labels"
TRAINING = GOLD / "training"
METADATA = GOLD / "metadata"

OUTPUT_TABLES = ROOT / "eda" / "gold" / "outputs" / "tables" / NOTEBOOK_NAME
OUTPUT_CHARTS = ROOT / "eda" / "gold" / "outputs" / "charts" / NOTEBOOK_NAME
OUTPUT_REPORTS = ROOT / "eda" / "gold" / "outputs" / "reports" / NOTEBOOK_NAME
INSIGHTS = ROOT / "eda" / "gold" / "insights"
CHECKPOINTS = ROOT / "eda" / "gold" / "checkpoints"
for path in [OUTPUT_TABLES, OUTPUT_CHARTS, OUTPUT_REPORTS, INSIGHTS, CHECKPOINTS]:
    path.mkdir(parents=True, exist_ok=True)

def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

def write_report(name: str, payload: dict) -> None:
    (OUTPUT_REPORTS / f"{name}.json").write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")

def write_insight(title: str, observations: list[str], issues: list[str], recommendations: list[str]) -> None:
    content = f"# {title}\n\n"
    content += f"**Generated at:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    content += "## Key Observations\n\n" + "\n".join(f"- {item}" for item in observations) + "\n\n"
    content += "## Issues\n\n" + ("\n".join(f"- {item}" for item in issues) if issues else "- None") + "\n\n"
    content += "## Recommendations\n\n" + "\n".join(f"- {item}" for item in recommendations) + "\n"
    (INSIGHTS / f"{NOTEBOOK_NAME}.md").write_text(content, encoding="utf-8")

def save_chart(fig, name: str) -> None:
    fig.write_html(OUTPUT_CHARTS / f"{name}.html", include_plotlyjs="cdn")

print("=" * 72)
print(f"GOLD EDA - {NOTEBOOK_NAME}")
print(f"Start time: {datetime.now()}")
print(f"Gold root: {GOLD}")


GOLD EDA - 04_horizon_dataset_audit
Start time: 2026-06-02 14:28:49.102473
Gold root: D:\F1_WinRate_Predictor\data\gold


## Horizon Row Coverage

The horizon datasets are different modeling views over the same Gold master
frame. Fixed horizons support comparable snapshots, while live-any-lap supports
interactive race-state prediction.

In [2]:
training_contract = read_json(METADATA / "training_dataset_contract.json")
dataset_rows = []
for name, meta in training_contract["datasets"].items():
    dataset_rows.append({
        "dataset": name,
        "rows": int(meta["rows"]),
        "columns": int(meta["columns"]),
        "unique_sessions": int(meta["unique_sessions"]),
        "unique_driver_sessions": int(meta["unique_driver_sessions"]),
    })
dataset_summary = pd.DataFrame(dataset_rows).sort_values("dataset")
dataset_summary.to_csv(OUTPUT_TABLES / "horizon_dataset_summary.csv", index=False)

fig = px.bar(
    dataset_summary,
    x="dataset",
    y="rows",
    color="dataset",
    text="rows",
    title="Gold Training Dataset Row Coverage by Horizon",
    labels={"dataset": "Training dataset", "rows": "Rows"},
)
fig.update_layout(showlegend=False, margin=dict(l=10, r=10, t=55, b=100))
fig.update_traces(texttemplate="%{text:,}", textposition="outside", cliponaxis=False)
save_chart(fig, "horizon_row_coverage")
fig.show()

display(dataset_summary)

,dataset,rows,columns,unique_sessions,unique_driver_sessions
1,horizon_25,1328,67,68,1328
2,horizon_50,1328,67,68,1328
3,horizon_75,1328,67,68,1328
4,horizon_panel,3984,67,68,1328
0,live_any_lap,62422,65,68,1328


## Target Balance by Horizon

Class balance should be stable across fixed horizons because the target is a
driver-session outcome. Material drift here would suggest horizon selection is
dropping a non-random subset of drivers.

In [3]:
dist_rows = []
for name, meta in training_contract["datasets"].items():
    path = ROOT / meta["artifact"]
    df = pd.read_parquet(path)
    if "target_finish_bucket_label" not in df.columns:
        continue
    counts = df.groupby("target_finish_bucket_label").size().rename("rows").reset_index()
    counts["dataset"] = name
    counts["pct"] = (counts["rows"] / counts["rows"].sum() * 100).round(2)
    dist_rows.append(counts)
target_distribution = pd.concat(dist_rows, ignore_index=True)
target_distribution.to_csv(OUTPUT_TABLES / "horizon_target_distribution.csv", index=False)

fig = px.bar(
    target_distribution,
    x="dataset",
    y="rows",
    color="target_finish_bucket_label",
    title="Target Distribution Across Gold Horizon Datasets",
    labels={"dataset": "Dataset", "rows": "Rows", "target_finish_bucket_label": "Finish bucket"},
)
fig.update_layout(margin=dict(l=10, r=10, t=55, b=100))
save_chart(fig, "horizon_target_distribution")
fig.show()

display(target_distribution.sort_values(["dataset", "target_finish_bucket_label"]))

,target_finish_bucket_label,rows,dataset,pct
5,CLASSIFIED_OUTSIDE_POINTS,562,horizon_25,42.32
6,DNF_DNS_DSQ_UNCLASSIFIED,116,horizon_25,8.73
7,PODIUM_NON_WIN,136,horizon_25,10.24
8,POINTS_NON_PODIUM,446,horizon_25,33.58
9,WIN,68,horizon_25,5.12
10,CLASSIFIED_OUTSIDE_POINTS,562,horizon_50,42.32
11,DNF_DNS_DSQ_UNCLASSIFIED,116,horizon_50,8.73
12,PODIUM_NON_WIN,136,horizon_50,10.24
13,POINTS_NON_PODIUM,446,horizon_50,33.58
14,WIN,68,horizon_50,5.12


In [4]:
fixed_ok = dataset_summary[dataset_summary["dataset"].isin(["horizon_25", "horizon_50", "horizon_75"])]["unique_driver_sessions"].nunique() == 1
write_report("horizon_dataset_audit", {
    "datasets": dataset_summary.to_dict(orient="records"),
    "fixed_horizon_driver_session_consistency": bool(fixed_ok),
})
write_insight(
    "Gold Horizon Dataset Audit",
    [
        "Fixed horizons keep one selected row per driver-session.",
        "Live-any-lap keeps many rows per driver-session for interactive prediction.",
        "Target distribution is expected to remain stable across fixed horizons.",
    ],
    [] if fixed_ok else ["Fixed horizons do not share the same driver-session coverage."],
    [
        "Use fixed horizons for comparable race-progress experiments.",
        "Use live_any_lap for Streamlit-style lap slider prediction and larger training volume.",
    ],
)
(CHECKPOINTS / "gold_horizon_dataset_audit_completed.txt").write_text(datetime.now().isoformat(), encoding="utf-8")

26